In [1]:
import numpy as np
import pandas as pd

# Baseline Recommender - Top N recommendations
This notebook is used to explore a top $N$ recommender as the baseline recommender, simply recommending the $N$ most bought items in the train data. A choice for $N$ could be $N=6$, since 6 items are displayed at a time in a row on the [Instacart website](https://www.instacart.com/store/walmart/storefront).

In [2]:
# load train data (prior)
df = pd.read_csv("data/order_products__prior.csv")

# grouping by product_id and counting the occurrence of each product across all orders
grouped_df = df.groupby(["product_id"])["product_id"].size().reset_index(name="product_id_count")

# sorting by count in descending order
sorted_df = grouped_df.sort_values("product_id_count", ascending=False)
print(sorted_df)

       product_id  product_id_count
24848       24852            472565
13172       13176            379450
21133       21137            264683
21899       21903            241921
47198       47209            213584
...           ...               ...
45884       45893                 1
13393       13397                 1
30445       30451                 1
42456       42464                 1
25244       25248                 1

[49677 rows x 2 columns]


In [3]:
# retrieving the top_n most bough products
top_n = 6
top_n_products = sorted_df.iloc[:6, :]["product_id"]
print(top_n_products)

24848    24852
13172    13176
21133    21137
21899    21903
47198    47209
47755    47766
Name: product_id, dtype: int64


In [4]:
orders_df = pd.read_csv("data/orders.csv")
orders_df

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,2539329,1,prior,1,2,8,NaN
1,2398795,1,prior,2,3,7,15.0
2,473747,1,prior,3,3,12,21.0
3,2254736,1,prior,4,4,7,29.0
4,431534,1,prior,5,4,15,28.0
...,...,...,...,...,...,...,...
3421078,2266710,206209,prior,10,5,18,29.0
3421079,1854736,206209,prior,11,4,10,30.0
3421080,626363,206209,prior,12,1,12,18.0
3421081,2977660,206209,prior,13,1,12,7.0


In [5]:
val_df = orders_df[orders_df["eval_set"] == "train"]   # contains only unique user_ids
val_df

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
10,1187899,1,train,11,4,8,14.0
25,1492625,2,train,15,1,11,30.0
49,2196797,5,train,5,0,11,6.0
74,525192,7,train,21,2,11,6.0
78,880375,8,train,4,1,14,10.0
...,...,...,...,...,...,...,...
3420838,2585586,206199,train,20,2,16,30.0
3420862,943915,206200,train,24,6,19,6.0
3420924,2371631,206203,train,6,4,19,30.0
3420933,1716008,206205,train,4,1,16,10.0


In [6]:
test_df = orders_df[orders_df["eval_set"] == "test"]   # contains only unique user_ids
test_df

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
38,2774568,3,test,13,5,15,11.0
44,329954,4,test,6,3,12,30.0
53,1528013,6,test,4,3,16,22.0
96,1376945,11,test,8,6,11,8.0
102,1356845,12,test,6,1,20,30.0
...,...,...,...,...,...,...,...
3420918,2728930,206202,test,23,2,17,6.0
3420929,350108,206204,test,5,4,14,14.0
3421001,1043943,206206,test,68,0,20,0.0
3421018,2821651,206207,test,17,2,13,14.0


In [7]:
order_products_train_df = pd.read_csv("data/order_products__train.csv")
order_products_train_df

,order_id,product_id,add_to_cart_order,reordered
0,1,49302,1,1
1,1,11109,2,1
2,1,10246,3,0
3,1,49683,4,0
4,1,43633,5,1
...,...,...,...,...
1384612,3421063,14233,3,1
1384613,3421063,35548,4,1
1384614,3421070,35951,1,1
1384615,3421070,16953,2,1


In [11]:
val_products_df = order_products_train_df.merge(val_df, on="order_id", how="left")[["product_id", "user_id"]]

In [14]:
val_products_df[val_products_df["user_id"] == 2]

,product_id,user_id
606747,22963,2
606748,7963,2
606749,16589,2
606750,32792,2
606751,41787,2
606752,22825,2
606753,13640,2
606754,24852,2
606755,45066,2
606756,9387,2


In [12]:
user_product_dict = val_products_df.groupby("user_id")["product_id"].apply(list).to_dict()
user_product_dict

{1: [196,
  25133,
  38928,
  26405,
  39657,
  10258,
  13032,
  26088,
  27845,
  49235,
  46149],
 2: [22963,
  7963,
  16589,
  32792,
  41787,
  22825,
  13640,
  24852,
  45066,
  9387,
  5450,
  24838,
  38547,
  19019,
  12007,
  26352,
  22559,
  45613,
  31883,
  12324,
  33957,
  5699,
  31612,
  34284,
  48523,
  2361,
  48821,
  11913,
  45645,
  1757,
  21329],
 5: [15349, 19057, 16185, 21413, 20843, 20114, 48204, 40706, 21616],
 7: [12053, 47272, 37999, 13198, 43967, 40852, 17638, 29894, 45066],
 8: [15937,
  5539,
  10960,
  23165,
  22247,
  4853,
  27104,
  7058,
  41259,
  37803,
  48230,
  47766,
  31717,
  21903,
  25659,
  41540,
  48121,
  2846],
 9: [27555,
  42347,
  27596,
  8834,
  26604,
  12075,
  8467,
  38988,
  30252,
  18926,
  24954,
  40571,
  1559,
  33754,
  29594,
  17600,
  42828,
  10132,
  20899,
  27973,
  41844,
  30967],
 10: [29650, 48720, 24654, 10177],
 13: [27435, 27086, 4210, 47078, 19934],
 14: [11042,
  32115,
  28601,
  29615,
  15869